In [1]:
"""
This file only works for 3D slice by slice inference.
"""

import os
import glob
import sys
from pathlib import Path

sys.path.append("../..")
#import opensimplex

#from torchvision.utils import save_image

import matplotlib.pyplot as plt
import numpy as np
import json
import argparse
import csv
import torch
import torch.nn.functional as F
from monai import transforms
from monai.data import CacheDataset, DataLoader
from monai.utils import set_determinism, StrEnum
from torch.amp import autocast
from tqdm import tqdm

import nibabel as nib

from monai.networks.schedulers import DDPMScheduler

from typing import Union

import pandas as pd

import AnoDDPM.simplex as simplex

import utils.custom_transforms as custom_transforms

import utils.simplex_ddpm as simplex_ddpm
import utils.thor_ddpm as thor_ddpm
from utils.utils import *

from monai.metrics import compute_iou, DiceMetric


from scipy.ndimage import median_filter, binary_erosion, binary_dilation
from multiprocessing import Pool, cpu_count
from functools import partial
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp

DEVICE_TYPE = "cuda:0"

In [4]:
ROOT_DIR = "/bettik/PROJECTS/pr-gin5_aini/fehrdelt/"
#ROOT_DIR = "/home/fehrdelt/bettik/"

In [5]:
config_dict = json.load(open(ROOT_DIR+"AnoDiffExperiments/experiment_2/exp_2_2/config.json", "r"))
args = argparse.Namespace(**config_dict)


In [11]:
EXPERIMENT_NAME = args.experiment_name
SUB_EXPERIMENT_NAME = args.sub_experiment_name
SUB_EXPERIMENT_DIR = f"{ROOT_DIR}/AnoDiffExperiments/{EXPERIMENT_NAME}/{SUB_EXPERIMENT_NAME}/"


In [13]:
ANOMALY_MAPS_DIR_SELECT_PARAMS = ROOT_DIR+f"datasets/anomaly_maps/{SUB_EXPERIMENT_NAME}_select_params/"
ANOMALY_MAPS_DIR = ROOT_DIR+f"datasets/anomaly_maps/{SUB_EXPERIMENT_NAME}/" # final anomaly maps with best params
os.makedirs(ANOMALY_MAPS_DIR_SELECT_PARAMS, exist_ok=True)
os.makedirs(ANOMALY_MAPS_DIR, exist_ok=True)

In [7]:
NOISE_MIN = int(args.compute_metrics_reconstruction["noise_rate_min"]*args.noise["num_timesteps_full_noise"])
NOISE_MAX = int(args.compute_metrics_reconstruction["noise_rate_max"]*args.noise["num_timesteps_full_noise"])+1
NOISE_INTERVAL = int(args.compute_metrics_reconstruction["noise_timesteps_interval"])

In [8]:
num_timesteps_to_try = np.arange(NOISE_MIN, NOISE_MAX, NOISE_INTERVAL)
thresholds_to_try = np.arange(0.0, 0.2, 0.02) # from 0.0 to 0.2 with step 0.02
median_filter_sizes_to_try = [-1, 3, 5] # -1 means no median filter
erosion_dilation_iterations_to_try = [0, 1, 2]

In [90]:

def process_anomaly_file(anomaly_file, anomaly_maps_folder, masks_folder, 
                             thresholds_to_try, median_filter_sizes_to_try, 
                             erosion_dilation_iterations_to_try):
    """Process a single anomaly file and return scores for all parameter combinations."""
    
    dm = DiceMetric(reduction="sum")

    local_iou_scores = {}
    local_dice_scores = {}
    
    
    #tprint(f"starting processing {anomaly_file}")

    anomaly_map_nib = nib.load(os.path.join(anomaly_maps_folder, anomaly_file))
    anomaly_map = anomaly_map_nib.get_fdata()
    timesteps = int(anomaly_file.split('.')[0].split('_')[-1])

    mask_nib = nib.load(os.path.join(masks_folder, f"{anomaly_file.split('_')[0]}.nii.gz"))
    mask = torch.from_numpy(mask_nib.get_fdata())
    mask = mask.unsqueeze(0).unsqueeze(0)  # B1HWD
    
    # Pre-compute filtered versions
    filtered_maps = {-1: torch.from_numpy(anomaly_map)}
    for median_filter_size in median_filter_sizes_to_try:
        if median_filter_size > 0:
            filtered_np = median_filter(anomaly_map, size=median_filter_size)
            filtered_maps[median_filter_size] = torch.from_numpy(filtered_np)
    
    
    #tprint(f"computed median filtered for {anomaly_file}")

    # Iterate through all combinations efficiently
    for median_filter_size in median_filter_sizes_to_try:
        final_anomaly_map = filtered_maps[median_filter_size]
        
        
        #tprint(f"median filter size {median_filter_size} for {anomaly_file}")
        #tprint(f"next: thresholds to try: {thresholds_to_try}")

        for threshold in thresholds_to_try:
            ano_segmentation_base = (final_anomaly_map > threshold)

            
            #tprint(f"threshold {threshold} for {anomaly_file}")
            
            for erosion_dilation_iterations in erosion_dilation_iterations_to_try:

                
                #tprint(f"erosion_dilation {erosion_dilation_iterations} for {anomaly_file}")

                if erosion_dilation_iterations > 0:
                    ano_segmentation_np = ano_segmentation_base.cpu().numpy()
                    
                    ano_segmentation_np = binary_erosion(ano_segmentation_np, iterations=erosion_dilation_iterations)
                    ano_segmentation_np = binary_dilation(ano_segmentation_np, iterations=erosion_dilation_iterations)

                    ano_segmentation = torch.from_numpy(ano_segmentation_np)
                else:
                    ano_segmentation = ano_segmentation_base

                #tprint(f"computing iou score for {anomaly_file} ..")
                # ano_segmentation and masks must be in format : B1HWD
                ano_segmentation = ano_segmentation.unsqueeze(0).unsqueeze(0)
                

                # Compute metrics
                iou_score = compute_iou(ano_segmentation, mask)
                flattened_iou_score = iou_score.cpu().numpy().flatten()
                flattened_iou_score = flattened_iou_score[~np.isnan(flattened_iou_score)]

                #tprint(f"computing dice score for {anomaly_file} ..")
                
                dice_score = dm(ano_segmentation, mask).cpu().numpy().flatten()
                dice_score = dice_score[~np.isnan(dice_score)]


                # Store results
                idx = (timesteps, threshold, median_filter_size, erosion_dilation_iterations)
                local_iou_scores[idx] = np.sum(flattened_iou_score)
                local_dice_scores[idx] = np.sum(dice_score)
    
    #tprint(f"finished for {anomaly_file} ..")

    return local_iou_scores, local_dice_scores

In [91]:

# Create the MultiIndex from timesteps, thresholds, median filter sizes, erosion and dilation iterations
iou_scores_midx = pd.MultiIndex.from_product([num_timesteps_to_try, thresholds_to_try, median_filter_sizes_to_try, erosion_dilation_iterations_to_try])
iou_scores_df = pd.DataFrame(index=iou_scores_midx, columns=["IOU"])
iou_scores_df.fillna(0.0, inplace=True)

dice_scores_midx = pd.MultiIndex.from_product([num_timesteps_to_try, thresholds_to_try, median_filter_sizes_to_try, erosion_dilation_iterations_to_try])
dice_scores_df = pd.DataFrame(index=dice_scores_midx, columns=["DICE"])
dice_scores_df.fillna(0.0, inplace=True)


Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


In [92]:
dice_scores_df.head()

DICE
50 0.0 -1 0   0.0
          1   0.0
          2   0.0
        3 0   0.0
          1   0.0

In [93]:
dice_scores_df.to_csv(SUB_EXPERIMENT_DIR+"test_dice_scores.csv")

In [94]:
anomaly_maps_folder = ANOMALY_MAPS_DIR_SELECT_PARAMS+"large/"
masks_folder = ROOT_DIR+"datasets/final_soop_dataset_small/masks_combined_registered/"
total_nb_images = len(os.listdir(anomaly_maps_folder))

In [95]:

anomaly_files = [entry.name for entry in os.scandir(anomaly_maps_folder) if entry.is_file() and entry.name.endswith(".nii.gz")][:1]
if not anomaly_files:
    raise RuntimeError(f"No anomaly map files found in '{anomaly_maps_folder}'.")

In [96]:
print(anomaly_files)

['sub-190_t_300.nii.gz']


In [97]:


process_func = partial(
    process_anomaly_file,
    anomaly_maps_folder=anomaly_maps_folder,
    masks_folder=masks_folder,
    thresholds_to_try=thresholds_to_try,
    median_filter_sizes_to_try=median_filter_sizes_to_try,
    erosion_dilation_iterations_to_try=erosion_dilation_iterations_to_try,
)

max_workers = min(32, mp.cpu_count())
ctx = mp.get_context("spawn")

In [98]:


if len(anomaly_files) == 1 or max_workers == 1:
    results = [process_func(file_name) for file_name in anomaly_files]
else:
    results = []
    with ProcessPoolExecutor(max_workers=max_workers, mp_context=ctx) as executor:
        futures = {executor.submit(process_func, file_name): file_name for file_name in anomaly_files}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing anomaly maps"):
            results.append(future.result())


In [99]:
print(results)

[({(300, 0.0, -1, 0): 0.02144654180499146, (300, 0.0, -1, 1): 0.02418890691393644, (300, 0.0, -1, 2): 0.0329278712382629, (300, 0.02, -1, 0): 0.06548698841934524, (300, 0.02, -1, 1): 0.07446623129762339, (300, 0.02, -1, 2): 0.11539847833617971, (300, 0.04, -1, 0): 0.07269952538810752, (300, 0.04, -1, 1): 0.09590687311139158, (300, 0.04, -1, 2): 0.15026843519825023, (300, 0.06, -1, 0): 0.07818146238414025, (300, 0.06, -1, 1): 0.11071853175792072, (300, 0.06, -1, 2): 0.12481846262302021, (300, 0.08, -1, 0): 0.08098028098374938, (300, 0.08, -1, 1): 0.11134279704493828, (300, 0.08, -1, 2): 0.08639428690573753, (300, 0.1, -1, 0): 0.08181975845609801, (300, 0.1, -1, 1): 0.10120245656609642, (300, 0.1, -1, 2): 0.05241143026093197, (300, 0.12, -1, 0): 0.07981977813687463, (300, 0.12, -1, 1): 0.07734713380864441, (300, 0.12, -1, 2): 0.020622652171509263, (300, 0.14, -1, 0): 0.07547612225874537, (300, 0.14, -1, 1): 0.05295843797870484, (300, 0.14, -1, 2): 0.012075321306710522, (300, 0.16, -1, 0)

Probleme : ça me sort plein de dice scores en liste pour chaque volume et je sais pas par combien diviser pour avoir un truc entre 0 et 1

**Est ce que c'est parce que le DiceMetric de MONAI s'attend à avoir B,H,W mais moi je donne HWD ?**

In [100]:
total_nb_images = 1

In [101]:
# Aggregate results from all processes
for local_iou_scores, local_dice_scores in results:
    for idx, iou_val in local_iou_scores.items():
        if np.isnan(iou_scores_df.loc[idx, "IOU"]):
            iou_scores_df.loc[idx, "IOU"] = iou_val
            dice_scores_df.loc[idx, "DICE"] = local_dice_scores[idx]
        else:
            iou_scores_df.loc[idx, "IOU"] += iou_val
            dice_scores_df.loc[idx, "DICE"] += local_dice_scores[idx]

# Divide everything by the number of images (moved outside the loop)
iou_scores_df = iou_scores_df / total_nb_images
dice_scores_df = dice_scores_df / total_nb_images

In [102]:
print(total_nb_images)

1


In [103]:
dice_scores_df.head()

DICE
50 0.0 -1 0   0.0
          1   0.0
          2   0.0
        3 0   0.0
          1   0.0

In [105]:
dice_scores_df.index.names = ['timesteps', 'threshold', 'median_filter_size', 'erosion_dilation_iterations']
iou_scores_df.index.names = ['timesteps', 'threshold', 'median_filter_size', 'erosion_dilation_iterations']

In [106]:
dice_scores_df.to_csv(SUB_EXPERIMENT_DIR+"test_dice_scores_filled.csv")